# 20 — Do festival-calendar features improve next-week surge ranking?

**Optional research after a completed 09b direct-history Optuna study.** Use the verified Python 3.12 project kernel and **Restart Kernel and Run All**. No other research notebook is required, including the wholesale audit. Existing 2024 caches and prepared data are reused; no downloads occur.

Compare the **frozen 09b Optuna control** with the same LightGBM settings plus 12 calendar flags. Training/evaluation rows, original 22 features, 10% labels and coverage policies stay identical. No parameter search, source corrections or dashboard replacement occurs. The calendar is a known-in-principle scheduling input, not future observed prices or outcomes.

Order: verify control → declare calendar and features → reconstruct matching training → compare the three expanding folds → record the fold conclusion → evaluate January–March 2026 once → save both positive and negative findings. These periods have already been inspected, so all new findings remain exploratory. No April–June test or registered November outcome is read.

**Timing limitation:** these are retrospective scheduled-date features. Original as-of calendar publications have not been archived in this project; timely historical availability is assumed, not independently certified. Lunar dates may have been provisional. No last-minute additional holidays or substitute days are included. A gain would support further investigation, not establish causal festival effects or independent future accuracy.

## 1. Verify the fixed control

Use the latest completed 09b direct-history Optuna study, as in research 15/16. This is **not** the dashboard's inherited tuned checkpoint or research 17's 200-proposal candidate. The experiment isolates a feature change against this named control; comparisons against different settings are separate questions. Parameters remain fixed before later validation is accessed.

In [ ]:
import sys
from pathlib import Path
REPOSITORY_ROOT = Path.cwd().resolve()
if not (REPOSITORY_ROOT / "pricecatcher_config.py").is_file():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent
if not (REPOSITORY_ROOT / "pricecatcher_config.py").is_file():
    raise FileNotFoundError("Open this notebook inside the FYP repository.")
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

import ast
import hashlib
import json
from datetime import datetime, timezone
from importlib import metadata, reload
from time import perf_counter
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (average_precision_score, confusion_matrix, precision_recall_curve,
                             precision_score, recall_score, f1_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from lightgbm import LGBMClassifier
import pricecatcher_config as cfg
cfg = reload(cfg)

def file_hash(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

history_runs = sorted((cfg.PROJECT_ROOT / "09b_longer_history_lightgbm").glob("*/tuned/study_manifest.json"), reverse=True)
if not history_runs:
    raise FileNotFoundError("Complete 09b_longer_history_lightgbm.ipynb first.")
control_dir = history_runs[0].parent.parent / "optuna"
if not (control_dir / "study_manifest.json").is_file():
    raise FileNotFoundError("The latest 09b run needs its completed direct-history Optuna result.")
parent = cfg.verify_saved_study(control_dir)
selection = parent["selection"]
if selection["family"] != "LightGBM" or selection["training_start"] != "2024-01":
    raise ValueError("Expected the completed longer-history LightGBM control.")
if sys.version_info[:2] != tuple(map(int, parent.get("python_version", "3.12").split(".")[:2])):
    raise ValueError("Use the same Python major/minor version as 09b.")
for package in ["lightgbm", "scikit-learn", "numpy", "pandas", "joblib"]:
    if metadata.version(package) != parent["package_versions"][package]:
        raise ValueError(f"Use the saved 09b {package} version for this controlled comparison.")

FOLDS = ["fold_1", "fold_2", "fold_3"]
folders = {name: cfg.PROJECT_ROOT / "experiments" / name / "05_modelling" for name in FOLDS + ["holdout"]}
settings_by_fold = {name: json.loads((folder / "experiment_settings.json").read_text()) for name, folder in folders.items()}
settings = settings_by_fold["holdout"]
if any(settings["preparation"][key] != value for key, value in cfg.PREPARATION_SETTINGS.items()):
    raise ValueError("Current configuration differs from the saved preparation. Restore its matching configuration.")
FEATURES, NUMERIC, CATEGORICAL = (settings[key] for key in ["feature_columns", "numeric_features", "categorical_features"])
if FEATURES != selection["feature_columns"] or settings["surge_threshold_pct"] != 10:
    raise ValueError("Keep the control feature list and 10% definition unchanged.")
MODEL_PARAMETERS = {**selection["fixed_parameters"], **selection["parameters"],
                    "random_state": selection["seed"], "n_jobs": selection["n_jobs"]}
control_model = joblib.load(control_dir / "lightgbm.joblib")
fitted_parameters = control_model.named_steps["model"].get_params()
if any(fitted_parameters[key] != value for key, value in MODEL_PARAMETERS.items()):
    raise ValueError("Declared model settings differ from the saved control model.")
parent_manifest_sha256 = file_hash(control_dir / "study_manifest.json")
configuration_sha256 = file_hash(Path(cfg.__file__))
print("Active run:", cfg.PROJECT_ROOT)
print("Frozen control:", control_dir)
print("Parameters (unchanged):", MODEL_PARAMETERS)

## 2. Declare the calendar and 12 flags before fitting

Use the **first day** of Chinese New Year, Hari Raya Aidilfitri, Deepavali and Christmas as an anchor. This is festival context across all states, **not a state-specific public-holiday or shop-closure calendar**. Deepavali is not a public holiday in every state. Festival effects are hypotheses, not assumed price rises.

For a row whose completed origin week starts on Monday *t*, the forecast week begins *t + 7 days*. Each festival has three binary flags: forecast week is the week **before**, **containing**, or **after** its anchor. Weeks are Monday–Sunday. No second-day, substitute-day, extra-holiday, weather or price information is inferred.

Dates were cross-checked against the Malaysia calendar in `holidays` **0.108**, using Kuala Lumpur and `observed=False`; that package is **not a new project dependency**. The readable table is embedded below. Calendar coverage is restricted to the current experiment's origin dates (2024 through March 2026).

Reference implementation: [Malaysia calendar](https://github.com/vacanza/holidays/blob/ee645d96ea71cada9f09d1ec76351d29a2433a9c/holidays/countries/malaysia.py). It lists official annual calendar sources, including [2024](https://www.kabinet.gov.my/bkpp/pdf/hari_kelepasan_am/hka_2024.pdf), [2025](https://www.kabinet.gov.my/bkpp/pdf/hari_kelepasan_am/hka_2025.pdf) and [2026](https://www.kabinet.gov.my/storage/2025/08/HKA-2026.pdf). Those official PDFs and historical publication vintages were **not independently retrieved for this implementation**. Check them before claiming an archived real-time calendar evaluation; the reference source retrieved now is retrospective. The additional 20 March 2026 holiday is deliberately excluded.

In [ ]:
FESTIVALS = ["cny", "aidilfitri", "deepavali", "christmas"]
PHASES = {"before": -7, "during": 0, "after": 7}
CALENDAR_ROWS = [
    ("cny", "2024-02-10"), ("aidilfitri", "2024-04-10"),
    ("deepavali", "2024-10-31"), ("christmas", "2024-12-25"),
    ("cny", "2025-01-29"), ("aidilfitri", "2025-03-31"),
    ("deepavali", "2025-10-20"), ("christmas", "2025-12-25"),
    ("cny", "2026-02-17"), ("aidilfitri", "2026-03-21"),
    ("deepavali", "2026-11-08"), ("christmas", "2026-12-25"),
]
calendar = pd.DataFrame(CALENDAR_ROWS, columns=["festival", "anchor_date"])
calendar["anchor_date"] = pd.to_datetime(calendar.anchor_date)
calendar["anchor_week"] = calendar.anchor_date - pd.to_timedelta(calendar.anchor_date.dt.dayofweek, unit="D")
if calendar.duplicated(["festival", "anchor_date"]).any():
    raise ValueError("Duplicate calendar anchors.")
EXTRA_FEATURES = [f"{festival}_{phase}" for festival in FESTIVALS for phase in PHASES]
FORMULAS = {name: "forecast-week Monday equals anchor-week Monday plus phase offset"
            for name in EXTRA_FEATURES}
CALENDAR_POLICY = {"reference": "holidays 0.108, MY/KUL, observed=False; dates transcribed, no runtime dependency",
    "reference_source": "https://github.com/vacanza/holidays/blob/ee645d96ea71cada9f09d1ec76351d29a2433a9c/holidays/countries/malaysia.py",
    "anchor": "festival first day; Monday-Sunday calendar week", "phase_offsets_days": PHASES,
    "forecast_week": "origin week_start + 7 days", "state_specific_holidays": False,
    "extra_or_substitute_holidays": False, "historical_asof_publications_archived": False,
    "scope": "retrospective exploratory scheduled-date representation"}
feature_support_rows = []


def calendar_features(frame):
    origin = pd.to_datetime(frame.week_start, errors="raise")
    if (origin.isna().any() or origin.dt.tz is not None
            or not origin.dt.dayofweek.eq(0).all() or not origin.eq(origin.dt.normalize()).all()
            or not origin.between("2024-01-01", "2026-03-31").all()):
        raise ValueError("Expected timezone-naive origin Mondays in 2024 through March 2026.")
    target_week = origin + pd.Timedelta(days=7)
    result = pd.DataFrame(index=frame.index)
    for festival in FESTIVALS:
        anchors = calendar.loc[calendar.festival.eq(festival), "anchor_week"]
        for phase, offset in PHASES.items():
            result[f"{festival}_{phase}"] = target_week.isin(anchors + pd.Timedelta(days=offset)).astype("int8")
    return result


def record_calendar_support(frame, split):
    flags = calendar_features(frame)
    for feature in EXTRA_FEATURES:
        active = flags[feature].eq(1)
        feature_support_rows.append({"split": split, "feature": feature, "rows": len(frame),
            "active_rows": int(active.sum()), "active_origin_weeks": int(frame.loc[active, "week_start"].nunique()),
            "total_origin_weeks": int(frame.week_start.nunique()),
            "active_positive_rows": int(frame.loc[active, "surge_label"].sum())})


def model_inputs(frame, enhanced=False):
    X = frame[FEATURES].copy()
    if enhanced:
        X[EXTRA_FEATURES] = calendar_features(frame)
    for column in CATEGORICAL:
        X[column] = X[column].astype("string").fillna("__missing__").astype(str)
    return X


def fit_enhanced(training, evaluation):
    preprocess = ColumnTransformer([
        ("numeric", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True), NUMERIC + EXTRA_FEATURES),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=True), CATEGORICAL),
    ], sparse_threshold=1.0)
    model = Pipeline([("preprocess", preprocess), ("model", LGBMClassifier(**MODEL_PARAMETERS))])
    model.fit(model_inputs(training, enhanced=True), training.surge_label.astype(int))
    scores = model.predict_proba(model_inputs(evaluation, enhanced=True))[:, 1]
    if not np.isfinite(scores).all() or not ((scores >= 0) & (scores <= 1)).all():
        raise ValueError("Expected finite model scores in [0, 1].")
    return model, scores


def evaluate(y, scores, cutoff):
    y = np.asarray(y, dtype=int)
    alerts = np.asarray(scores) >= cutoff
    tn, fp, fn, tp = confusion_matrix(y, alerts, labels=[0, 1]).ravel()
    return {"rows": len(y), "positives": int(y.sum()), "average_precision":
        float(average_precision_score(y, scores)) if len(np.unique(y)) == 2 else np.nan,
        "cutoff": cutoff, "precision": precision_score(y, alerts, zero_division=0),
        "recall": recall_score(y, alerts, zero_division=0) if y.sum() else np.nan,
        "f1": f1_score(y, alerts, zero_division=0) if y.sum() else np.nan,
        "alerts": int(alerts.sum()), "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn)}


display(calendar)
print("Extra features:", EXTRA_FEATURES)
print("This fixed calendar is declared before any new fitting or validation comparison.")

## 3. Reuse 09b's verified training reconstruction

To avoid a second version of the historical-cleaning/weekly-label code, load **only the two named function definitions** from 09b. No notebook cells, downloads, tuning or deployment code are executed. As in 09b, original 2025 features/labels are reconstructed and checked before 2024 is added. The rebuilt counts must match this control's recorded populations.

The source functions, configuration and this notebook's calculation code are fingerprinted in the research manifest. The original model inputs come from saved preparation/history; the added flags come only from origin dates and the declared calendar. No observed future data enters those flags.

In [ ]:
history_notebook = REPOSITORY_ROOT / "09b_longer_history_lightgbm.ipynb"
source_nb = json.loads(history_notebook.read_text())
functions = []
for cell in source_nb["cells"]:
    if cell["cell_type"] == "code":
        tree = ast.parse("".join(cell["source"]))
        functions.extend(node for node in tree.body if isinstance(node, ast.FunctionDef)
                         and node.name in {"training_table", "prepare_training"})
if sorted(node.name for node in functions) != ["prepare_training", "training_table"]:
    raise ValueError("09b training helpers changed; review compatibility before this study.")
calculation = ast.Module(body=functions, type_ignores=[])
helper_sha256 = hashlib.sha256(ast.dump(calculation).encode()).hexdigest()
exec(compile(calculation, str(history_notebook), "exec"), globals())
helper_notebook_sha256 = file_hash(history_notebook)

additional_inputs = parent["additional_input_sha256"]
historical_paths = [cfg.resolve_input(path) for path in additional_inputs
                    if cfg.resolve_input(path).name == "selected_prices_2024.parquet"]
if len(historical_paths) != 1:
    raise ValueError("Expected the exact 2024 source snapshot recorded by 09b.")
earlier_source = pd.read_parquet(historical_paths[0], columns=cfg.GROUP_KEYS + ["date", "premise_code", "price"])
earlier_source["date"] = pd.to_datetime(earlier_source.date)
if earlier_source.empty or not earlier_source.date.between("2024-01-01", "2024-12-31").all():
    raise ValueError("Supplementary source must contain only the recorded 2024 observations.")
features, keys = FEATURES, cfg.GROUP_KEYS + ["week_start"]
training_rows, cleaning_rows = [], []
expected_support = pd.read_csv(control_dir / "training_comparison.csv")

def matching_training(name):
    training, evaluation = prepare_training(name)
    for history, recorded in [("2025-only", training_rows[-2]), ("2024-added", training_rows[-1])]:
        expected = expected_support.loc[expected_support.fold.eq(name) & expected_support.history.eq(history)]
        if len(expected) != 1:
            raise ValueError(f"{name}: missing original support evidence.")
        for column in ["rows", "positives", "first_week", "last_outcome"]:
            if str(recorded[column]) != str(expected.iloc[0][column]):
                raise ValueError(f"{name}: rebuilt {history} {column} differs from 09b.")
    if training.surge_label.nunique() != 2 or evaluation.surge_label.nunique() != 2:
        raise ValueError("Both classes are needed for the overall AP comparison.")
    return training, evaluation

## 4. Compare the three folds first

Fit once per fold with the 12 declared flags and unchanged control parameters. Reuse the verified control fold metrics on the same evaluation rows. Record support separately for training/evaluation: many district/item rows share one calendar week, so large active-row counts do not imply many independent festival events.

All three fold AP differences are reported, including declines. The feature set and windows are not changed after inspecting scores. No significance or causal claim follows from a positive mean. Features can also duplicate information already represented by month and seasonal week features.

In [ ]:
control_folds = pd.read_csv(control_dir / "fold_comparison.csv")
control_folds = control_folds.loc[control_folds.method.eq(selection["comparison_method"])].copy()
if len(control_folds) != 3 or set(control_folds.fold) != set(FOLDS):
    raise ValueError("Expected exactly three recorded control folds.")
control_summary = pd.read_csv(control_dir / "fold_summary.csv").set_index("method").loc[selection["comparison_method"]]
np.testing.assert_allclose([control_summary.mean_ap, control_summary.std_ap],
    [control_folds.average_precision.mean(), control_folds.average_precision.std()], rtol=1e-10, atol=1e-12)
fold_rows = []
for name in FOLDS:
    training, evaluation = matching_training(name)
    record_calendar_support(training, name + ": training")
    record_calendar_support(evaluation, name + ": evaluation")
    started = perf_counter()
    model, scores = fit_enhanced(training, evaluation)
    original = control_folds.set_index("fold").loc[name]
    if int(original.rows) != len(evaluation) or int(original.positives) != int(evaluation.surge_label.sum()):
        raise ValueError("Control and enhanced fold evaluation populations differ.")
    fold_rows.append({"fold": name, "training_rows": len(training), "evaluation_rows": len(evaluation),
        "control_ap": float(original.average_precision),
        "enhanced_ap": float(average_precision_score(evaluation.surge_label, scores)),
        "fit_and_score_seconds": perf_counter() - started})
    del training, evaluation, model, scores

fold_comparison = pd.DataFrame(fold_rows)
fold_comparison["ap_change"] = fold_comparison.enhanced_ap - fold_comparison.control_ap
fold_summary = pd.DataFrame([
    {"method": "Frozen 09b Optuna control", "features": len(FEATURES),
     "mean_fold_ap": fold_comparison.control_ap.mean(), "std_fold_ap": fold_comparison.control_ap.std()},
    {"method": "Add festival-calendar features", "features": len(FEATURES) + len(EXTRA_FEATURES),
     "mean_fold_ap": fold_comparison.enhanced_ap.mean(), "std_fold_ap": fold_comparison.enhanced_ap.std()},
])
fold_gain = float(fold_comparison.ap_change.mean())
fold_conclusion = "Higher mean fold AP" if fold_gain > 0 else "No improvement in mean fold AP"
display(fold_comparison)
display(fold_summary)
print(fold_conclusion, "| difference:", round(fold_gain, 6))
print("Fold conclusion frozen before Jan–Mar validation; this is exploratory evidence.")

## 5. Evaluate later validation once, including negative findings

Only after recording the fold conclusion, fit on the full longer-history training population and evaluate January–March 2026. The saved control's identifiers, labels and fresh scores must reproduce its recorded validation predictions. Both versions use identical rows.

Report AP, the common 0.5 cutoff and each model's own F1-selected cutoff. Cutoff-selection performance is apparent, not independent. Report group support and calendar-window coverage. Sparse events and repeated inspection limit generalisation; two training years do not establish robust festival effects.

In [ ]:
training, validation = matching_training("holdout")
prediction_keys = ["item_code", "state", "district", "premise_type", "project_group",
                   "week_start", "next_week_end", "surge_label"]
identifiers = prediction_keys[:-1]
validation = validation.sort_values(identifiers).reset_index(drop=True)
control_predictions = pd.read_parquet(control_dir / "validation_predictions.parquet").sort_values(identifiers).reset_index(drop=True)
if validation.duplicated(identifiers).any() or control_predictions.duplicated(identifiers).any():
    raise ValueError("Duplicate evaluation identifiers.")
pd.testing.assert_frame_equal(validation[prediction_keys], control_predictions[prediction_keys])
control_scores = control_model.predict_proba(model_inputs(validation))[:, 1]
np.testing.assert_allclose(control_scores, control_predictions.model_score, rtol=1e-9, atol=1e-11)
record_calendar_support(training, "holdout: training")
record_calendar_support(validation, "holdout: validation")
enhanced_model, enhanced_scores = fit_enhanced(training, validation)
del training, control_model

p, r, cutoffs = precision_recall_curve(validation.surge_label, enhanced_scores)
f1 = np.divide(2*p[:-1]*r[:-1], p[:-1]+r[:-1], out=np.zeros(len(cutoffs)), where=(p[:-1]+r[:-1]) > 0)
enhanced_cutoff = float(cutoffs[np.flatnonzero(np.isclose(f1, f1.max(), rtol=0, atol=1e-12))[-1]])
score_sets = {"Frozen 09b Optuna control": (control_scores, float(selection["score_cutoff"])),
              "Add festival-calendar features": (enhanced_scores, enhanced_cutoff)}
validation_rows, group_rows = [], []
predictions = validation[prediction_keys].copy()
for method, (scores, cutoff) in score_sets.items():
    for role, threshold in [("common 0.5 reference", 0.5), ("apparent cutoff-selection performance", cutoff)]:
        validation_rows.append({"method": method, "role": role, **evaluate(validation.surge_label, scores, threshold)})
    for group, positions in validation.groupby("project_group", sort=True).indices.items():
        group_rows.append({"method": method, "project_group": group,
                           **evaluate(validation.iloc[positions].surge_label, scores[positions], cutoff)})
    prefix = "control" if method.startswith("Frozen") else "enhanced"
    predictions[prefix + "_score"] = scores
    predictions[prefix + "_alert"] = scores >= cutoff
validation_comparison = pd.DataFrame(validation_rows)
validation_groups = pd.DataFrame(group_rows)
display(validation_comparison)
display(validation_groups)
validation_gain = float(average_precision_score(validation.surge_label, enhanced_scores)
                        - average_precision_score(validation.surge_label, control_scores))
print("Later validation AP difference:", round(validation_gain, 6))
print("Neither period is an independent final test; no dashboard replacement is made.")


## 6. Save the isolated research evidence

Save the calendar, feature support, paired metrics/predictions, model and provenance in one separate timestamped folder. No deployment pin or main comparison is rewritten. The completed manifest is written last. The saved research model needs the declared calendar-feature function when used for prediction; it is not a drop-in dashboard replacement.

In [ ]:
# Verify upstream bytes again before publishing a completed study.
cfg.verify_saved_study(control_dir)
if (file_hash(control_dir / "study_manifest.json") != parent_manifest_sha256
        or file_hash(history_notebook) != helper_notebook_sha256
        or file_hash(Path(cfg.__file__)) != configuration_sha256):
    raise ValueError("Parent study or calculation code changed during this experiment.")
this_notebook = REPOSITORY_ROOT / "research" / "20_festival_calendar_experiment.ipynb"
notebook = json.loads(this_notebook.read_text())
code_sha256 = hashlib.sha256("\n".join("".join(c["source"]) for c in notebook["cells"]
                                       if c["cell_type"] == "code").encode()).hexdigest()
output = cfg.PROJECT_ROOT / "research" / "20_festival_calendar_experiment" / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
output.mkdir(parents=True, exist_ok=False)
for name, table in {"fold_comparison.csv": fold_comparison, "fold_summary.csv": fold_summary,
    "validation_comparison.csv": validation_comparison, "validation_group_metrics.csv": validation_groups,
    "training_support.csv": pd.DataFrame(training_rows), "historical_cleaning.csv": pd.DataFrame(cleaning_rows),
    "festival_calendar.csv": calendar, "calendar_feature_support.csv": pd.DataFrame(feature_support_rows)}.items():
    table.to_csv(output / name, index=False)
predictions.to_parquet(output / "validation_predictions.parquet", index=False)
joblib.dump(enhanced_model, output / "research_model.joblib")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fold_comparison.set_index("fold")[["control_ap", "enhanced_ap"]].plot.bar(ax=axes[0], rot=0)
for method, (scores, _) in score_sets.items():
    precision, recall, _ = precision_recall_curve(validation.surge_label, scores)
    axes[1].plot(recall, precision, label=method)
axes[0].set(ylabel="Average precision", ylim=(0, 1), title="Expanding folds (exploratory)")
axes[1].axhline(validation.surge_label.mean(), color="grey", linestyle=":", label="Positive fraction")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Jan–Mar validation (already inspected)")
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(output / "comparison.png", dpi=150); plt.show()
protocol = {"status": "completed", "question": "twelve festival-calendar flags with fixed longer-history LightGBM settings",
    "python_version": sys.version.split()[0], "parameters": MODEL_PARAMETERS,
    "feature_columns": FEATURES + EXTRA_FEATURES, "extra_feature_formulas": FORMULAS,
    "surge_threshold_pct": 10.0, "fold_conclusion": fold_conclusion,
    "mean_fold_ap_change": fold_gain, "validation_ap_change": validation_gain,
    "score_cutoff": enhanced_cutoff, "score_interpretation": "uncalibrated model score",
    "calendar_policy": CALENDAR_POLICY, "calendar_rows": CALENDAR_ROWS,
    "independent_test_evaluated": False, "dashboard_changed": False,
    "selection_rule": "fixed feature experiment; fold conclusion before later validation; no hyperparameter search",
    "validation_role": "already-inspected exploratory comparison and F1 cutoff selection",
    "parent_study": cfg.input_reference(control_dir / "study_manifest.json"),
    "parent_manifest_sha256": parent_manifest_sha256,
    "input_sha256": parent["input_sha256"], "additional_input_sha256": parent["additional_input_sha256"],
    "historical_helper_sha256": helper_sha256, "configuration_sha256": configuration_sha256,
    "research_code_sha256": code_sha256,
    "package_versions": {p: metadata.version(p) for p in ["lightgbm", "scikit-learn", "numpy", "pandas", "joblib"]},
    "output_sha256": {p.name: file_hash(p) for p in output.iterdir()}}
(output / "research_manifest.json").write_text(json.dumps(protocol, indent=2))
print("Saved:", output)
print("Share fold_comparison.csv, validation_comparison.csv and calendar_feature_support.csv first.")

## 7. Interpret the result without changing the goalposts

- If fold and later AP improve, describe a promising exploratory calendar contribution, requiring independent confirmation. Report whether gains occur across folds/groups, not just overall.
- If only later validation improves, the result does not overturn the previously recorded fold conclusion. Do not redesign flags around that validation period.
- If AP does not improve, retain the negative result; seasonality may already be captured, the model may not use sparse flags, or effects may differ across time/groups. These are possibilities, not established explanations.
- Improved AP and a worse alert trade-off can coexist. Compare false alerts, missed surges and group support at the declared cutoffs.
- A forecast-week calendar flag is permissible in principle because dates can be scheduled in advance; it must not be confused with knowing next week's prices. Original as-of date vintages were not independently verified here.
- Even a gain would not resolve missing weather/fuel/import inputs, source-price uncertainty or demonstrated affordability impact. The dashboard remains pinned unless separately and deliberately changed.

The proposed 12-feature block is tested once as declared. Do not run repeated alternative windows until validation rises, redefine labels to inflate AP, or treat many rows sharing a festival week as independent repeated events.